# AprilTag mount calibration

The URDF carries the two AprilTag mount transforms (`volcaniarm_apriltag.xacro`): the base tag on `volcaniarm_base_link` and the end-effector tag on `right_arm_tip_link`. While these translations are placeholders, every accuracy figure contains a constant fiducial offset that has nothing to do with the arm. This notebook solves for corrections to both mount translations from multi-pose run data and prints ready-to-paste xacro origin lines.

**How it works.** For each observation the base-relative residual `r = (det_ee - det_base) - (urdf_ee - urdf_base)` is formed in world Y-Z. To first order `r = A(phi) d - b`, where `d` is the EE mount correction in the tip frame, `b` the base mount correction in world Y-Z, and `A(phi)` the in-plane rotation of the tip at that pose. Stacking observations gives a linear least-squares problem.

Two properties make this practical:

- **Camera translation immunity.** Both detections pass through the same world-to-camera transform, so a camera translation error cancels exactly in `det_ee - det_base`. The mounts can be solved with an uncalibrated camera pose. Camera rotation does not cancel; a nuisance-term cross-check below quantifies it.
- **Observability.** Two poses with different tip angles already make the 4-unknown system full rank, and the workspace spans roughly 70 degrees of tip rotation. A single-goal dataset is degenerate (only `A d - b` is observable), which is why this needs workspace-coverage data, not static-accuracy data at one goal.

**Workflow**: run camera localization, sweep the workspace grid (1 run suffices, 2 to 3 preferred), solve here, edit the xacro, rebuild, **re-run camera localization** (its old solution absorbed the old EE mount error), then verify with a 30-cycle static accuracy run: the mean residual should land within about 2 mm of zero.

*Note: the kernel imports `volcaniarm_calibration` through a `.pth` file; restart the kernel after changing the package code.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'figure.dpi': 110,
    'font.size': 13, 'axes.titlesize': 15, 'axes.titleweight': 'bold',
    'axes.labelsize': 13, 'axes.edgecolor': '#999999', 'axes.grid': True,
    'grid.color': '#e8e8e8', 'grid.linewidth': 0.8,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
})
PRIMARY = '#2b6cb0'
ACCENT = '#2e9c4a'
# Fixed categorical order for runs, assigned by run order and kept
# stable across every figure in the notebook.
RUN_COLORS = ['#2b6cb0', '#2e9c4a', '#c79a3a', '#8b5cb0',
              '#d04b4b', '#3a9ea6', '#b0642b', '#666666']


def per_axis_residuals_mm(frame):
    """Camera-independent per-axis residuals in world Y-Z (mm).

    Uses the base-relative vector so a camera translation error
    cancels: r = (det_ee - det_base) - (urdf_ee - urdf_base).
    """
    ry = ((frame['det_ee_y'] - frame['det_base_y'])
          - (frame['urdf_ee_y'] - frame['urdf_base_y'])) * 1000.0
    rz = ((frame['det_ee_z'] - frame['det_base_z'])
          - (frame['urdf_ee_z'] - frame['urdf_base_z'])) * 1000.0
    return ry.to_numpy(), rz.to_numpy()

def run_short(run_id):
    # 'static_accuracy/2026-07-08/17-57-39' -> '07-08 17:57'
    parts = str(run_id).split('/')
    if len(parts) == 3:
        return parts[1][5:] + ' ' + parts[2][:5].replace('-', ':')
    return str(run_id)

from volcaniarm_calibration.analysis import (
    load_runs, select_comparable_runs, concat_runs,
    solve_mounts, suggest_xacro, tip_angle_from_thetas,
)
from volcaniarm_calibration.analysis.mounts import (
    R_WORLD_BASELINK_DEFAULT, _rot_from_quat, _rx,
)

# Pool every completed multi-pose run recorded with the current mounts.
# Static-accuracy runs at different goals can be added to the pool; the
# solver only needs pose diversity, not a particular test type.
TEST_NAMES = ['workspace_coverage']
RUN_DIRS = None
ALLOW_MOUNT_KEYS = None

runs = []
for name in TEST_NAMES:
    runs += load_runs(name)
runs = select_comparable_runs(runs, allow_mount_keys=ALLOW_MOUNT_KEYS,
                              match_goals=False)
HAVE_DATA = bool(runs)
if HAVE_DATA:
    df = concat_runs(runs)
    n_goals = df[['goal_y', 'goal_z']].drop_duplicates().shape[0]
    print(f'Runs: {len(runs)}, observations: {len(df)}, '
          f'distinct goals: {n_goals}')
    if n_goals < 2:
        print('WARNING: only one distinct goal; the solve is degenerate. '
              'Record a workspace sweep first.')
else:
    df = None
    print('No completed multi-pose runs on disk. Run workspace_coverage '
          'from the dashboard first.')

## Tip-angle source check

The solver needs the tip rotation per observation. New runs record it from TF (`tip_q*` columns); for legacy runs a pure-python mirror of the five-bar forward kinematics reconstructs it from the commanded joint angles. Where both are available they are compared; a large discrepancy would indicate the kinematic mirror is out of sync with the URDF.

In [ ]:
if HAVE_DATA:
    phi = df.apply(lambda r: tip_angle_from_thetas(
        r['theta_left'], r['theta_right']), axis=1).to_numpy()
    phi_deg = np.degrees(phi)
    have_q = df['tip_qw'].notna().to_numpy()
    if have_q.any():
        max_dev = 0.0
        for idx in np.flatnonzero(have_q):
            row = df.iloc[idx]
            A_q = _rot_from_quat(row['tip_qx'], row['tip_qy'],
                                 row['tip_qz'], row['tip_qw'])[1:3, 1:3]
            A_k = (R_WORLD_BASELINK_DEFAULT @ _rx(phi[idx]))[1:3, 1:3]
            max_dev = max(max_dev, float(np.abs(A_q - A_k).max()))
        print(f'{have_q.sum()}/{len(df)} rows carry recorded tip '
              f'quaternions; max rotation-matrix deviation vs the '
              f'kinematic mirror: {max_dev:.4f}')
    else:
        print('No recorded tip quaternions (legacy runs); using the '
              'kinematic mirror for all rows.')
    print(f'Tip angle span: {np.nanmin(phi_deg):.1f} to '
          f'{np.nanmax(phi_deg):.1f} deg')
    fig, ax = plt.subplots(figsize=(7.0, 3.6))
    ax.hist(phi_deg[~np.isnan(phi_deg)], bins=15, color=PRIMARY,
            edgecolor='white')
    ax.set_xlabel('tip in-plane angle  [deg]')
    ax.set_ylabel('observations')
    ax.set_title('Pose diversity available to the solver')
    fig.tight_layout()

## Solve

Least-squares solve for the four unknowns. The condition number reflects the pose diversity (a well-spread sweep gives order 10 or less); the residual RMS before vs after states how much of the observed error the two mount corrections explain.

In [ ]:
COND_LIMIT = 100.0  # above this the EE/base split is unreliable

if HAVE_DATA:
    sol = solve_mounts(df)
    print(sol)
    if sol.cond > COND_LIMIT:
        print()
        print('WARNING: condition number is large; the dataset lacks '
              'tip-angle diversity, so the split between the EE and '
              'base corrections is unreliable even though the fit '
              'residual looks small. Record a workspace sweep and do '
              'NOT apply these values.')

## Camera-rotation cross-check

The same solve with one extra nuisance unknown: an in-plane camera rotation. One degree of camera roll maps to roughly 12 mm at the 0.7 m tag separation, so if the two solves disagree materially the stand should be checked for level before the mount values are trusted.

In [ ]:
if HAVE_DATA:
    sol_cam = solve_mounts(df, estimate_camera_rotation=True)
    print(sol_cam)
    shift_mm = 1000.0 * max(
        float(np.abs(sol_cam.d_tip_yz - sol.d_tip_yz).max()),
        float(np.abs(sol_cam.b_world_yz - sol.b_world_yz).max()))
    omega_deg = np.degrees(sol_cam.omega_rad)
    print()
    if sol.cond > COND_LIMIT:
        print('Cross-check skipped as pass/fail: the primary solve is '
              'ill-conditioned (see the warning above).')
    elif abs(omega_deg) > 0.3 or shift_mm > 2.0:
        print(f'WARNING: solves disagree (mount shift {shift_mm:.2f} mm, '
              f'camera rotation {omega_deg:+.2f} deg). Verify the camera '
              f'stand is level and repeat the sweep before applying.')
    else:
        print(f'Consistent: mount shift {shift_mm:.2f} mm, camera '
              f'rotation {omega_deg:+.2f} deg. Safe to apply.')

## Residuals before and after

Per-observation residual magnitude against the tip angle. Structure remaining in the after-fit residuals as a function of the tip angle indicates error the mount model cannot absorb (link lengths, camera rotation).

In [ ]:
if HAVE_DATA:
    ry, rz = per_axis_residuals_mm(df)
    before_mm = np.hypot(ry, rz)
    # solve_mounts drops NaN rows internally; rebuild its mask here so
    # the before/after arrays align.
    needed = ['det_base_y', 'det_base_z', 'det_ee_y', 'det_ee_z',
              'urdf_base_y', 'urdf_base_z', 'urdf_ee_y', 'urdf_ee_z',
              'theta_left', 'theta_right']
    mask = df[needed].notna().all(axis=1).to_numpy()
    phi_used = np.degrees(df.apply(lambda r: tip_angle_from_thetas(
        r['theta_left'], r['theta_right']), axis=1).to_numpy())[mask]

    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    ax.plot(phi_used, before_mm[mask], 'o', ms=8, color='#aaaaaa',
            mec='white', mew=0.8, label='before correction')
    ax.plot(phi_used, sol.per_row_residual_mm, 'o', ms=8, color=PRIMARY,
            mec='white', mew=0.8, label='after correction')
    ax.set_xlabel('tip in-plane angle  [deg]')
    ax.set_ylabel('residual magnitude  [mm]')
    ax.set_title('Mount-model residual vs arm configuration')
    ax.legend(loc='best', framealpha=0.9)
    fig.tight_layout()

## Suggested xacro values

In [ ]:
if HAVE_DATA:
    if sol.cond > COND_LIMIT:
        print('Not printing xacro values: the solve is ill-conditioned '
              '(insufficient tip-angle diversity).')
    else:
        print(suggest_xacro(sol))
        print('File to edit:')
        print('  src/volcaniarm_description/urdf/volcaniarm_apriltag.xacro')

## Applying the correction

1. Paste the origin lines above into `volcaniarm_apriltag.xacro` (only the `xyz` values change; keep the rpy).
2. `colcon build --symlink-install --packages-select volcaniarm_description`, then relaunch the bringup.
3. **Re-run Camera Localization from the dashboard.** The previous camera solution absorbed the old EE mount error and is now stale.
4. Verify: one 30-cycle static accuracy run. Acceptance: `|mean d_error| < 2 mm`. If a few millimetres remain, one more sweep-solve-apply iteration converges.
5. Update the mirrored mount constants at the top of `analysis/loader.py` (flagged as a manual sync) so legacy-run tooling agrees with the URDF.

Runs recorded before and after the edit carry different mount versions in `config.yaml` and are never averaged together by the notebooks.

## Troubleshooting

- **Large condition number (over ~100)**: insufficient tip-angle spread. Add the grid corner goals and re-sweep; do not solve from single-goal data.
- **rms_after well above the detector noise (a few mm)**: the mount model cannot explain the data. Candidates: camera in-plane rotation (see the cross-check), five-bar link-length error, a tag physically loose on its bracket.
- **Cross-check disagrees**: level the camera stand, re-run camera localization, re-sweep.
- **Solution changes between sweeps**: check the tag brackets and that `tag_size` matches the printed tags.